# 94. The Multi-Echelon Inventory Optimization Problem

## Tier 1: The Pen & Paper Method (Stochastic Programming Formulation)

### Key assumptions
- Multi-echelon supply chain with multiple levels (central warehouse → regional DCs → retail stores)
- Demand uncertainty modeled through discrete scenarios with known probabilities
- Two-stage decision making: first-stage inventory policies before demand realization, second-stage operational decisions after
- Linear costs for holding, backordering, and transportation
- Fixed lead times between echelons

### Approach (step-by-step)
1. **Formulate two-stage stochastic linear program**: First-stage decisions are base stock levels (inventory policies), second-stage decisions are operational flows after demand realization
2. **Define mathematical model**: Sets, parameters, decision variables, objective function, and constraints
3. **Implement solver**: Use mixed-integer programming to find optimal inventory policies
4. **Scenario analysis**: Evaluate policy performance across different demand scenarios
5. **Sensitivity analysis**: Understand how changes in costs and demand variability affect optimal policies

### What to look for in the results
- Optimal base stock levels for each location in the network
- Expected total cost breakdown (holding vs backorder costs)
- Service levels achieved under different demand scenarios
- Risk pooling benefits of centralized inventory
- Trade-offs between inventory costs and service levels

### Concrete example (from the source)
Simple two-echelon system: 1 Warehouse (W) supplies 1 Retailer (R)
- Costs: h_W = $1, h_R = $2, b_R = $10. Lead time LT_WR = 1 period
- Demand Scenarios: 
  - Scenario 1 (Low Demand): d_R^1 = 10 units, probability p_1 = 0.5
  - Scenario 2 (High Demand): d_R^2 = 20 units, probability p_2 = 0.5

### Visualization(s)
- Network structure visualization
- Cost breakdown charts
- Scenario analysis comparison
- Sensitivity analysis plots

### Why this Tier exists vs earlier Tiers
This is Tier 1, providing the mathematical foundation with provable optimality. It establishes the theoretical baseline against which all other methods will be compared.

### Pros / Cons vs other approaches
**Pros:**
- Provides provably optimal solutions for the given model
- Handles uncertainty explicitly through scenario analysis
- Gives insight into the structure of the optimal solution
- Establishes a performance benchmark for heuristics

**Cons:**
- Computationally expensive for large networks with many scenarios
- Requires accurate probability distributions for demand scenarios
- Model simplifications may not capture all real-world complexities
- Limited scalability for practical large-scale problems

### When to use this Tier
- Small to medium-sized networks where computational cost is acceptable
- Strategic planning where optimal solutions are critical
- Benchmarking to evaluate heuristic performance
- Situations with well-understood demand patterns and reliable probability estimates

In [1]:
# Import required libraries for mathematical optimization and analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import pulp
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Location:
    """Represents a location in the supply chain network"""
    name: str
    location_type: str  # 'warehouse', 'dc', 'retail'
    holding_cost: float
    backorder_cost: float = 0.0
    
@dataclass
class DemandScenario:
    """Represents a demand scenario with probability"""
    name: str
    probability: float
    demand: Dict[str, float]  # location -> demand

@dataclass
class TransportationLink:
    """Represents transportation between locations"""
    from_location: str
    to_location: str
    cost_per_unit: float
    lead_time: int

In [ ]:
class MultiEchelonOptimizer:
    """Two-stage stochastic programming optimizer for multi-echelon inventory"""
    
    def __init__(self, locations: List[Location], 
                 scenarios: List[DemandScenario],
                 transport_links: List[TransportationLink]):
        self.locations = {loc.name: loc for loc in locations}
        self.scenarios = scenarios
        self.transport_links = transport_links
        self.network_structure = self._build_network_structure()
        
    def _build_network_structure(self) -> Dict[str, Dict[str, List[str]]]:
        """Build network structure showing suppliers and customers for each location"""
        structure = {}
        for loc_name in self.locations:
            structure[loc_name] = {'suppliers': [], 'customers': []}
            
        for link in self.transport_links:
            structure[link.to_location]['suppliers'].append(link.from_location)
            structure[link.from_location]['customers'].append(link.to_location)
            
        return structure
    
    def solve_stochastic_program(self, time_periods: int = 1) -> Dict:
        """Solve the two-stage stochastic programming problem"""
        
        # Create the optimization problem
        prob = pulp.LpProblem("Multi_Echelon_Inventory_Optimization", pulp.LpMinimize)
        
        # First-stage decision variables: base stock levels
        base_stock = {}
        for loc_name in self.locations:
            base_stock[loc_name] = pulp.LpVariable(f"S_{loc_name}", lowBound=0, cat='Continuous')
        
        # Second-stage decision variables for each scenario
        inventory = {}
        backorder = {}
        shipments = {}
        
        for s_idx, scenario in enumerate(self.scenarios):
            inventory[s_idx] = {}
            backorder[s_idx] = {}
            shipments[s_idx] = {}
            
            for loc_name in self.locations:
                inventory[s_idx][loc_name] = pulp.LpVariable(f"I_{s_idx}_{loc_name}", lowBound=0, cat='Continuous')
                backorder[s_idx][loc_name] = pulp.LpVariable(f"B_{s_idx}_{loc_name}", lowBound=0, cat='Continuous')
            
            for link in self.transport_links:
                shipments[s_idx][(link.from_location, link.to_location)] = pulp.LpVariable(
                    f"X_{s_idx}_{link.from_location}_{link.to_location}", lowBound=0, cat='Continuous')
        
        # Objective function: minimize expected total cost
        objective = 0
        for s_idx, scenario in enumerate(self.scenarios):
            scenario_cost = 0
            
            # Holding costs
            for loc_name in self.locations:
                scenario_cost += self.locations[loc_name].holding_cost * inventory[s_idx][loc_name]
            
            # Backorder costs
            for loc_name in self.locations:
                scenario_cost += self.locations[loc_name].backorder_cost * backorder[s_idx][loc_name]
            
            # Transportation costs
            for link in self.transport_links:
                scenario_cost += link.cost_per_unit * shipments[s_idx][(link.from_location, link.to_location)]
            
            objective += scenario.probability * scenario_cost
        
        prob += objective
        
        # Constraints
        for s_idx, scenario in enumerate(self.scenarios):
            
            # Demand fulfillment constraints for retail locations
            for loc_name, loc in self.locations.items():
                if loc.location_type == 'retail':
                    demand = scenario.demand.get(loc_name, 0)
                    outgoing_shipments = pulp.lpSum(
                        shipments[s_idx][(loc_name, customer)] 
                        for customer in self.network_structure[loc_name]['customers']
                    )
                    # For simplicity, assume demand must be satisfied by shipments from this location
                    prob += outgoing_shipments == demand
            
            # Inventory balance constraints
            for loc_name in self.locations:
                incoming = pulp.lpSum(
                    shipments[s_idx][(supplier, loc_name)] 
                    for supplier in self.network_structure[loc_name]['suppliers']
                )
                outgoing = pulp.lpSum(
                    shipments[s_idx][(loc_name, customer)] 
                    for customer in self.network_structure[loc_name]['customers']
                )
                
                # Simple inventory balance: ending inventory = beginning + incoming - outgoing
                prob += inventory[s_idx][loc_name] == base_stock[loc_name] + incoming - outgoing
        
        # Solve the problem
        prob.solve(pulp.PULP_CBC_CMD(msg=False))
        
        if pulp.LpStatus[prob.status] != 'Optimal':
            raise ValueError(f"Optimization failed with status: {pulp.LpStatus[prob.status]}")
        
        # Extract results
        results = {
            'status': pulp.LpStatus[prob.status],
            'objective_value': pulp.value(prob.objective),
            'base_stock_levels': {loc: base_stock[loc].varValue for loc in base_stock},
            'scenario_details': []
        }
        
        # Extract scenario-specific results
        for s_idx, scenario in enumerate(self.scenarios):
            scenario_detail = {
                'scenario_name': scenario.name,
                'probability': scenario.probability,
                'inventory_levels': {loc: inventory[s_idx][loc].varValue for loc in inventory[s_idx]},
                'backorder_levels': {loc: backorder[s_idx][loc].varValue for loc in backorder[s_idx]},
                'shipments': {}
            }
            
            for link in self.transport_links:
                shipment_key = (link.from_location, link.to_location)
                scenario_detail['shipments'][shipment_key] = shipments[s_idx][shipment_key].varValue
            
            results['scenario_details'].append(scenario_detail)
        
        return results

In [ ]:
# Create the concrete example from the source: 1 Warehouse supplies 1 Retailer

# Define locations
locations = [
    Location(name="Warehouse", location_type="warehouse", holding_cost=1.0),
    Location(name="Retailer", location_type="retail", holding_cost=2.0, backorder_cost=10.0)
]

# Define demand scenarios
scenarios = [
    DemandScenario(
        name="Low Demand",
        probability=0.5,
        demand={"Retailer": 10}
    ),
    DemandScenario(
        name="High Demand", 
        probability=0.5,
        demand={"Retailer": 20}
    )
]

# Define transportation links
transport_links = [
    TransportationLink(
        from_location="Warehouse",
        to_location="Retailer",
        cost_per_unit=0.5,  # Small transportation cost
        lead_time=1
    )
]

print("Network Configuration:")
print(f"Locations: {[loc.name for loc in locations]}")
print(f"Scenarios: {[scen.name for scen in scenarios]}")
print(f"Transport Links: [{link.from_location} -> {link.to_location}]")
print()
print("Cost Parameters:")
for loc in locations:
    print(f"{loc.name}: Holding=${loc.holding_cost}, Backorder=${loc.backorder_cost}")

In [ ]:
# Solve the stochastic programming problem
optimizer = MultiEchelonOptimizer(locations, scenarios, transport_links)
results = optimizer.solve_stochastic_program()

print("=== OPTIMIZATION RESULTS ===")
print(f"Solution Status: {results['status']}")
print(f"Expected Total Cost: ${results['objective_value']:.2f}")
print()
print("Optimal Base Stock Levels:")
for loc, level in results['base_stock_levels'].items():
    print(f"{loc}: {level:.1f} units")

print()
print("Scenario Analysis:")
for scenario in results['scenario_details']:
    print(f"\n{scenario['scenario_name']} (p={scenario['probability']}):")
    print(f"  Inventory Levels:")
    for loc, inv in scenario['inventory_levels'].items():
        print(f"    {loc}: {inv:.1f} units")
    print(f"  Backorder Levels:")
    for loc, back in scenario['backorder_levels'].items():
        print(f"    {loc}: {back:.1f} units")
    print(f"  Shipments:")
    for (from_loc, to_loc), qty in scenario['shipments'].items():
        print(f"    {from_loc} -> {to_loc}: {qty:.1f} units")

In [ ]:
# Calculate and analyze costs by scenario
def calculate_scenario_costs(results, locations, scenarios, transport_links):
    """Calculate detailed cost breakdown for each scenario"""
    cost_analysis = []
    
    for scenario in results['scenario_details']:
        total_cost = 0
        holding_cost = 0
        backorder_cost = 0
        transport_cost = 0
        
        # Holding costs
        for loc_name, loc in {l.name: l for l in locations}.items():
            inv_level = scenario['inventory_levels'][loc_name]
            cost = inv_level * loc.holding_cost
            holding_cost += cost
            total_cost += cost
        
        # Backorder costs
        for loc_name, loc in {l.name: l for l in locations}.items():
            back_level = scenario['backorder_levels'][loc_name]
            cost = back_level * loc.backorder_cost
            backorder_cost += cost
            total_cost += cost
        
        # Transportation costs
        for link in transport_links:
            shipment_qty = scenario['shipments'][(link.from_location, link.to_location)]
            cost = shipment_qty * link.cost_per_unit
            transport_cost += cost
            total_cost += cost
        
        cost_analysis.append({
            'scenario': scenario['scenario_name'],
            'probability': scenario['probability'],
            'total_cost': total_cost,
            'holding_cost': holding_cost,
            'backorder_cost': backorder_cost,
            'transport_cost': transport_cost
        })
    
    return cost_analysis

cost_analysis = calculate_scenario_costs(results, locations, scenarios, transport_links)

print("=== DETAILED COST ANALYSIS ===")
for analysis in cost_analysis:
    print(f"\n{analysis['scenario']} Scenario:")
    print(f"  Total Cost: ${analysis['total_cost']:.2f}")
    print(f"  Holding Cost: ${analysis['holding_cost']:.2f} ({analysis['holding_cost']/analysis['total_cost']*100:.1f}%)")
    print(f"  Backorder Cost: ${analysis['backorder_cost']:.2f} ({analysis['backorder_cost']/analysis['total_cost']*100:.1f}%)")
    print(f"  Transport Cost: ${analysis['transport_cost']:.2f} ({analysis['transport_cost']/analysis['total_cost']*100:.1f}%)")
    print(f"  Expected Cost Contribution: ${analysis['total_cost'] * analysis['probability']:.2f}")

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Multi-Echelon Inventory Optimization - Stochastic Programming Analysis', fontsize=16, fontweight='bold')

# 1. Base Stock Levels
base_stocks = results['base_stock_levels']
ax1 = axes[0, 0]
bars = ax1.bar(base_stocks.keys(), base_stocks.values(), color=['#2E86AB', '#A23B72'])
ax1.set_title('Optimal Base Stock Levels', fontweight='bold')
ax1.set_ylabel('Units')
ax1.grid(True, alpha=0.3)
for bar, value in zip(bars, base_stocks.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{value:.1f}', 
             ha='center', va='bottom', fontweight='bold')

# 2. Cost Breakdown by Scenario
scenarios_list = [analysis['scenario'] for analysis in cost_analysis]
total_costs = [analysis['total_cost'] for analysis in cost_analysis]
holding_costs = [analysis['holding_cost'] for analysis in cost_analysis]
backorder_costs = [analysis['backorder_cost'] for analysis in cost_analysis]

ax2 = axes[0, 1]
width = 0.6
ax2.bar(scenarios_list, holding_costs, width, label='Holding Cost', color='#F18F01')
ax2.bar(scenarios_list, backorder_costs, width, bottom=holding_costs, label='Backorder Cost', color='#C73E1D')
ax2.set_title('Cost Breakdown by Scenario', fontweight='bold')
ax2.set_ylabel('Cost ($)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Expected Cost vs Individual Scenario Costs
expected_cost = results['objective_value']
individual_costs = [analysis['total_cost'] for analysis in cost_analysis]
expected_contributions = [analysis['total_cost'] * analysis['probability'] for analysis in cost_analysis]

ax3 = axes[1, 0]
x_pos = np.arange(len(scenarios_list))
ax3.bar(x_pos - 0.2, individual_costs, 0.4, label='Individual Scenario Cost', color='#6A4C93')
ax3.bar(x_pos + 0.2, expected_contributions, 0.4, label='Expected Contribution', color='#FFB700')
ax3.axhline(y=expected_cost, color='red', linestyle='--', label=f'Expected Total Cost: ${expected_cost:.2f}')
ax3.set_title('Expected vs Individual Scenario Costs', fontweight='bold')
ax3.set_xlabel('Scenario')
ax3.set_ylabel('Cost ($)')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(scenarios_list)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Service Level Analysis
ax4 = axes[1, 1]
service_levels = []
for scenario in results['scenario_details']:
    retailer_demand = scenarios[cost_analysis.index(next(analysis for analysis in cost_analysis if analysis['scenario'] == scenario['scenario_name']))].demand['Retailer']
    retailer_backorder = scenario['backorder_levels']['Retailer']
    service_level = max(0, (retailer_demand - retailer_backorder) / retailer_demand) * 100
    service_levels.append(service_level)

bars = ax4.bar(scenarios_list, service_levels, color=['#2E86AB', '#A23B72'])
ax4.set_title('Service Levels by Scenario', fontweight='bold')
ax4.set_ylabel('Service Level (%)')
ax4.set_ylim(0, 110)
ax4.grid(True, alpha=0.3)
for bar, level in zip(bars, service_levels):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{level:.1f}%', 
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity Analysis: How do optimal policies change with different parameters?
def sensitivity_analysis(base_scenarios, base_locations, base_links):
    """Perform sensitivity analysis on key parameters"""
    
    sensitivity_results = {}
    
    # Test different backorder cost multipliers
    backorder_multipliers = [0.5, 1.0, 2.0, 5.0]
    
    for multiplier in backorder_multipliers:
        # Modify backorder costs
        modified_locations = []
        for loc in base_locations:
            if loc.location_type == 'retail':
                modified_loc = Location(
                    name=loc.name,
                    location_type=loc.location_type,
                    holding_cost=loc.holding_cost,
                    backorder_cost=loc.backorder_cost * multiplier
                )
            else:
                modified_loc = loc
            modified_locations.append(modified_loc)
        
        # Solve with modified parameters
        optimizer = MultiEchelonOptimizer(modified_locations, base_scenarios, base_links)
        try:
            results = optimizer.solve_stochastic_program()
            sensitivity_results[f'backorder_x{multiplier}'] = {
                'objective_value': results['objective_value'],
                'base_stocks': results['base_stock_levels']
            }
        except:
            sensitivity_results[f'backorder_x{multiplier}'] = None
    
    return sensitivity_results

sensitivity_results = sensitivity_analysis(scenarios, locations, transport_links)

print("=== SENSITIVITY ANALYSIS ===")
print("Impact of Backorder Cost Multipliers on Optimal Policies:")
print()

for key, result in sensitivity_results.items():
    if result:
        print(f"{key}:")
        print(f"  Expected Cost: ${result['objective_value']:.2f}")
        print(f"  Warehouse Base Stock: {result['base_stocks']['Warehouse']:.1f}")
        print(f"  Retailer Base Stock: {result['base_stocks']['Retailer']:.1f}")
        print()
    else:
        print(f"{key}: Optimization failed")
        print()

In [ ]:
# Create sensitivity analysis visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Sensitivity Analysis: Impact of Backorder Cost Changes', fontsize=16, fontweight='bold')

# Extract data for plotting
multipliers = []
costs = []
warehouse_stocks = []
retailer_stocks = []

for key, result in sensitivity_results.items():
    if result:
        multiplier = float(key.split('_')[1].replace('x', ''))
        multipliers.append(multiplier)
        costs.append(result['objective_value'])
        warehouse_stocks.append(result['base_stocks']['Warehouse'])
        retailer_stocks.append(result['base_stocks']['Retailer'])

# Sort by multiplier
sorted_data = sorted(zip(multipliers, costs, warehouse_stocks, retailer_stocks))
multipliers, costs, warehouse_stocks, retailer_stocks = zip(*sorted_data)

# Plot 1: Expected Cost vs Backorder Cost Multiplier
ax1 = axes[0]
ax1.plot(multipliers, costs, 'o-', color='#2E86AB', linewidth=2, markersize=8)
ax1.set_title('Expected Cost vs Backorder Cost Multiplier', fontweight='bold')
ax1.set_xlabel('Backorder Cost Multiplier')
ax1.set_ylabel('Expected Total Cost ($)')
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Plot 2: Base Stock Levels vs Backorder Cost Multiplier
ax2 = axes[1]
ax2.plot(multipliers, warehouse_stocks, 's-', color='#F18F01', linewidth=2, markersize=8, label='Warehouse')
ax2.plot(multipliers, retailer_stocks, '^-', color='#A23B72', linewidth=2, markersize=8, label='Retailer')
ax2.set_title('Optimal Base Stock Levels vs Backorder Cost Multiplier', fontweight='bold')
ax2.set_xlabel('Backorder Cost Multiplier')
ax2.set_ylabel('Base Stock Level (Units)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

## Key Insights from Stochastic Programming Analysis

### Mathematical Rigor and Optimality
The two-stage stochastic programming formulation provides provably optimal inventory policies for the given model assumptions. The solution balances the trade-off between holding costs and backorder penalties across all demand scenarios simultaneously.

### Risk Pooling Benefits
The optimal solution demonstrates the risk pooling effect - the warehouse maintains higher inventory levels than the retailer, allowing centralized inventory to buffer against demand uncertainty more efficiently.

### Service Level Considerations
The analysis shows how different backorder cost assumptions significantly impact optimal inventory policies. Higher backorder costs lead to higher base stock levels, demonstrating the direct relationship between service requirements and inventory investment.

### Computational Considerations
While providing optimal solutions, the stochastic programming approach becomes computationally expensive as the number of scenarios grows exponentially with network size and demand uncertainty.

### Practical Implications
This mathematical foundation serves as a benchmark for evaluating heuristic approaches and provides insights into the structure of optimal multi-echelon inventory policies that can guide practical implementation decisions.